# 14. Eine komplette Rechnung mit echten Zahlen

Aus *Transformer von Grund auf*.

In [ ]:
# In Colab liegt nichts auf der Festplatte, also holt diese Zelle
# die zwei oder drei Dateien, die das Kapitel braucht. Wer das
# Repository geklont hat, findet sie schon vor, und es wird
# nichts heruntergeladen.
import os
import sys
import urllib.request

BASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/notebooks/de/code/"
PLACES = [".", "code", "../code"]

for name in ["ch14_attention.py", "ch14_pytorch.py"]:
    if not any(os.path.exists(os.path.join(where, name))
               for where in PLACES):
        urllib.request.urlretrieve(BASE + name, name)

for where in PLACES:
    if os.path.isdir(where) and where not in sys.path:
        sys.path.insert(0, where)

Ein Attention-Head mit Zahlen, die du nachrechnen kannst.

Vier Tokens, eine Modelldimension von vier, ein Head, eine Key-Dimension von vier. Jedes Gewicht ist hier eine Null oder eine Eins, und jedes Embedding ist eine kleine ganze Zahl, also lässt sich jedes Produkt mit dem Bleistift nachrechnen. Die Scores werden durch die Wurzel aus vier geteilt, also genau durch zwei.

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True, floatmode="fixed")

TOKENS = ["die", "Katze", "trank", "Milch"]

# Eine Zeile pro Token. In einem trainierten Modell werden diese Werte
# gelernt. Hier sind sie so gewählt, dass "Katze" und "Milch" ein Merkmal
# teilen und "trank" nicht.
X = np.array([
    [1.0, 0.0, 1.0, 0.0],   # die
    [0.0, 2.0, 0.0, 1.0],   # Katze
    [1.0, 0.0, 2.0, 0.0],   # trank
    [0.0, 1.0, 0.0, 2.0],   # Milch
])

W_Q = np.array([
    [0.0, 0.0, 1.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
    [0.0, 1.0, 0.0, 0.0],
])

W_K = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
])

W_V = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [0.0, 0.0, 1.0, 0.0],
    [1.0, 0.0, 0.0, 1.0],
])

Softmax je Zeile, ums Zeilenmaximum verschoben: kein Überlauf.

In [ ]:
def softmax(scores):
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exponentiated = np.exp(shifted)
    return exponentiated / exponentiated.sum(axis=-1, keepdims=True)

Ein Attention-Head, jeder Zwischenschritt bleibt erhalten.

In [ ]:
def attention(x, w_q, w_k, w_v):
    queries = x @ w_q
    keys = x @ w_k
    values = x @ w_v
    d_k = keys.shape[-1]
    raw = queries @ keys.T
    scaled = raw / np.sqrt(d_k)
    weights = softmax(scaled)
    output = weights @ values
    return queries, keys, values, raw, scaled, weights, output

Eine beschriftete Matrix so ausgeben, wie das Buch sie druckt.

In [ ]:
def show(name, matrix, cols):
    print(f"\n{name}   Form {matrix.shape}")
    print(" " * 7 + "".join(f"{c:>8}" for c in cols))
    for label, row in zip(TOKENS, matrix):
        print(f"{label:>6} " + "".join(f"{v:>8.3f}" for v in row))

In [ ]:
def main():
    q, k, v, raw, scaled, weights, out = attention(X, W_Q, W_K, W_V)
    show("X, die Eingabe-Embeddings", X, ["e0", "e1", "e2", "e3"])
    show("Q = X W_Q", q, ["q0", "q1", "q2", "q3"])
    show("K = X W_K", k, ["k0", "k1", "k2", "k3"])
    show("V = X W_V", v, ["v0", "v1", "v2", "v3"])
    show("Rohe Scores, Q mal K transponiert", raw, TOKENS)
    show("Skalierte Scores, jeder Eintrag halbiert", scaled, TOKENS)
    show("Attention-Gewichte, Softmax je Zeile", weights, TOKENS)
    show("Ausgabe, Gewichte mal V", out, ["o0", "o1", "o2", "o3"])
    print("\nZeilensummen der Gewichte:", weights.sum(axis=1))

In [ ]:
main()

Jetzt die zweite Hälfte des Kapitels.

Derselbe Head in PyTorch, und der Beweis.

Worauf es in dieser Datei ankommt, sind die Assertions am Ende. Das Framework rechnet genau das, was ch14_attention.py rechnet, und nichts sonst, und glauben kann man das nur, wenn man beides laufen lässt und die Zahlen vergleicht.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

from ch14_attention import X, W_Q, W_K, W_V, attention

torch.set_printoptions(precision=3, sci_mode=False)

Ein Head, mit PyTorch-Operationen statt mit NumPy geschrieben.

In [ ]:
def attention_torch(x, w_q, w_k, w_v):
    queries = x @ w_q
    keys = x @ w_k
    values = x @ w_v
    d_k = keys.shape[-1]
    scores = queries @ keys.transpose(-2, -1) / d_k ** 0.5
    weights = torch.softmax(scores, dim=-1)
    return weights, weights @ values

In [ ]:
def main():
    x = torch.tensor(X, dtype=torch.float64)
    w_q = torch.tensor(W_Q, dtype=torch.float64)
    w_k = torch.tensor(W_K, dtype=torch.float64)
    w_v = torch.tensor(W_V, dtype=torch.float64)

    weights, output = attention_torch(x, w_q, w_k, w_v)
    print("Attention-Gewichte aus PyTorch")
    print(weights)
    print("\nAusgabe aus PyTorch")
    print(output)

    # derselbe Aufruf, übergeben an die eingebaute, fusionierte Version
    builtin = F.scaled_dot_product_attention(x @ w_q, x @ w_k, x @ w_v)

    _, _, _, _, _, np_weights, np_output = attention(X, W_Q, W_K, W_V)
    assert np.allclose(weights.numpy(), np_weights, atol=1e-12)
    assert np.allclose(output.numpy(), np_output, atol=1e-12)
    assert np.allclose(builtin.numpy(), np_output, atol=1e-12)
    print("\nNumPy, PyTorch und F.scaled_dot_product_attention"
          " stimmen überein")
    print("größte Abweichung:",
          float(np.abs(builtin.numpy() - np_output).max()))

In [ ]:
main()

Das Kapitel, aus dem dieses Notebook stammt, rechnet jede Zahl oben durch und erklärt, warum jeder Schritt da ist.